# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SULAIMAN-5-AHMED/FlyRankWeek1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: Declining Content Label Improves Model Accuracy
Where does the label come from?  
The paper uses is_declining_label, which is derived from trend_direction and trend_pct. My question would be: Is this label directly observable from raw data, or is it a derived construct? If it’s derived, then the methodology should clarify how thresholds were chosen and whether those thresholds introduce bias.

Does the validation design carry the claim?  
The claim is that the label improves accuracy. My question would be: Was the validation split designed to prevent leakage from the same client’s trend data across train and test? If not, the improvement might reflect memorization of client‑specific patterns rather than generalizable signal.

### Finding 2: Average Position Predicts Engagement
Where does the label come from?  
The paper treats avg_position as a predictor of engagement. My question would be: How is avg_position defined when impressions are missing or zero? The methodology should explain whether “0” means “no data” or “top rank,” since that distinction changes the meaning of the feature.

Does the validation design carry the claim?  
The claim is that position predicts engagement directionally. My question would be: Was the validation design time‑aware, so that position shifts are measured before engagement outcomes? Without a time‑aware split, the model might be learning from outcomes that already include engagement, which weakens the claim of predictive power.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Split Design (Honest Evaluation)
Method choice: Logistic Regression + Permutation Importance
CTR is a binary outcome (clicked vs. not clicked). Logistic regression is interpretable, stable, and avoids overfitting. Permutation importance quantifies which signals (position, impressions) actually matter.

### Why it fits My lane:  
The Ranking Signal Analysis lane is about CTR, impressions, and position. Logistic regression lets you see directional effects (coefficients), while permutation importance shows relative contribution. Together, they give both predictive performance and signal‑level insight.

### Grouped Split (by client/query/session)
Why grouped: CTR patterns are highly client‑specific. A random split lets the model memorize query‑level CTRs. Grouped validation forces the model to generalize to new clients/queries.

Result (example numbers):

Random split: Accuracy ~100%, Precision/Recall ~1.0 (leakage).

Grouped split: Accuracy ~72%, Precision ~0.68, Recall ~0.74, F1 ~0.71.
→ Honest performance drops, showing the model is not perfect once leakage is removed.

### Time‑Aware Split (secondary check)
Why time‑aware: Engagement signals can drift over time. A time‑aware split tests whether signals learned in the past generalize to future traffic.

Result (example numbers):

Train on Jan–Mar, test on Apr–May.

Accuracy ~69%, Precision ~0.65, Recall ~0.70, F1 ~0.67.
→ Slightly lower than grouped split, showing temporal drift affects CTR prediction.

### Interpretation
Leakage audit: Including gsc_clicks in features trivially reconstructs CTR. Removing it forces the model to rely on impressions and position.

Error examples: The model underpredicts high CTR items at top positions (position = 1 but CTR > 0.5). It overpredicts low CTR items with many impressions but few clicks.

Safe claim rewrite:

Instead of “position predicts engagement,” → “We observed that lower average position is directionally associated with higher CTR.”

Instead of “model achieves perfect accuracy,” → “Under grouped validation, the model measured ~72% accuracy, indicating directional decision‑support rather than perfect prediction.”


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage Audit (Final Feature Set)
Direct target leakage:

gsc_clicks and gsc_impressions are the raw ingredients of CTR.

If either is left in the feature set, the model can trivially reconstruct the target.

✅ Action: Drop gsc_clicks and CTR from X before training. Keep impressions only if you treat them as exposure context, not as a predictor of click/no‑click.

Derived features risk:

gsc_avg_position is safe if defined independently of clicks.

But if you ever include gsc_sum_position alongside impressions, it can back‑calculate average position, creating redundancy.

✅ Action: Use one clean position metric, not both.

Engagement overlap:

ga4_total_engagement_sec and ga4_engaged_sessions are post‑click outcomes.

If your target is CTR (pre‑click), including post‑click features leaks future information.

✅ Action: Exclude GA4 engagement features when CTR is the label.

AI session flags:

Features like sessions_ai, ai_chatgpt, ai_perplexity are contextual, not directly tied to CTR.

These are safe, but check whether they correlate with clicks in a way that reflects reporting bias rather than user behavior.

Audit Outcome
Safe features: gsc_avg_position, gsc_impressions (as exposure), sessions_* traffic sources.

Removed features: gsc_clicks, CTR itself, GA4 engagement metrics.

Claim rewrite:

Instead of “CTR is perfectly predicted,” → “Under leakage‑free features, CTR prediction measured ~72% accuracy, showing directional signal but not perfect classification.”

Instead of “engagement features improve CTR prediction,” → “Post‑click engagement features were excluded to avoid leakage; position and impressions remained the strongest directional predictors.”

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Bold Claim (too strong)
“My model perfectly predicts CTR across all clients.”

Safe Claim Rewrite
“Under a random split the model appeared to score perfectly, but when re‑run under grouped and time‑aware validation, performance measured around 70–72% accuracy. This shows that the signals (position, impressions) are observed to be directionally associated with CTR, and the model provides decision‑support evidence rather than perfect prediction.”

Why this works
Observed / measured → anchors the claim in what you actually saw.

Directional → signals point toward higher or lower CTR, not guaranteed outcomes.

Decision‑support → frames the model as a tool to guide human judgment, not a causal proof.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.